# Day 7: Event Operators — Why Markets Need a Groupoid, Not a Group

*Alpha Flow Research · HongJin HE · July 2026*

---

## The Problem: Discrete Events Break Continuous Models

Standard stochastic calculus handles *continuous* evolution. But markets are punctuated by **discrete events** that change the state space itself:

- **IPO**: universe grows from $n$ to $n+1$ assets
- **M&A merger**: two assets collapse to one — $n \to n-1$
- **Bankruptcy/delisting**: $n \to n-1$
- **Spin-off**: one asset becomes two — $n \to n+1$
- **Index rebalancing**: composition changes, total $n$ preserved

When $n$ changes, the operator $T_w: \mathbb{R}^n \to \mathbb{R}^n$ no longer has a well-defined composition. A semigroup requires a fixed domain — **we need a groupoid**.

## Three Event Modes

$$T_w(s) = A_w \cdot s + b_w + \Sigma_w \varepsilon_w, \quad \varepsilon_w \sim \mathcal{N}(0, I)$$

| Mode | Effect on $n$ | Example |
|---|---|---|
| **I — Local endomorphism** | $n \to n$ | Stock split, secondary offering |
| **II — Global tensor** | $n \to n$ | Rate hike, crisis |
| **III — Pairwise morphism** | $n \to n \pm 1$ | M&A merger, IPO, delisting |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sys
sys.path.insert(0, '..')

# Import our events module (events/operators.py)
try:
    from events.operators import (
        EventOperator, EventMode,
        stock_split_operator, rate_hike_operator,
        merger_operator, ipo_operator, compose
    )
    print('Events module imported successfully.')
    use_module = True
except ImportError:
    print('Running in standalone mode (events module not on path).')
    use_module = False

# Standalone demonstration
np.random.seed(2026)

d = 5  # state dimension per asset: [log_price, log_volume, volatility, sentiment, momentum]
n_assets = 4  # start with 4 assets

# Asset state: n_assets × d
S = np.random.randn(n_assets, d)
S[:, 0] += 5.0  # log prices around exp(5) ≈ $150
print(f'Initial universe: {n_assets} assets, state shape = {S.shape}')

# ── Mode I: Stock split 2:1 on asset 0 ──────────────────────────────────
def apply_stock_split(S, asset_idx, ratio):
    S_new = S.copy()
    S_new[asset_idx, 0] -= np.log(ratio)  # log price halves
    S_new[asset_idx, 1] += np.log(ratio)  # log volume doubles
    return S_new

S_after_split = apply_stock_split(S, 0, 2.0)
print(f'\nAfter 2:1 split on asset 0:')
print(f'  log_price: {S[0,0]:.3f} → {S_after_split[0,0]:.3f} (diff={S_after_split[0,0]-S[0,0]:.3f} ≈ -log(2))')

# ── Mode II: Rate hike (affects all assets) ──────────────────────────────
def apply_rate_hike(S, shock_bps):
    """Rate hike: reduces log prices by duration-weighted amount."""
    S_new = S.copy()
    rate_delta = shock_bps / 10000
    duration = np.array([7, 4, 2, 8])  # hypothetical durations
    S_new[:, 0] -= duration * rate_delta  # price impact
    S_new[:, 2] += 0.3 * rate_delta      # vol spike
    return S_new

S_after_hike = apply_rate_hike(S, 25)  # 25bp hike
print(f'\nAfter 25bp rate hike (Mode II — all assets):')
print(f'  Asset log prices change: {S_after_hike[:,0] - S[:,0]}')

# ── Mode III: M&A — assets 1 and 2 merge into asset 1 ───────────────────
def apply_merger(S, acquirer_idx, target_idx, exchange_ratio):
    """Merger: target absorbed by acquirer. Universe shrinks by 1."""
    S_new = np.delete(S, target_idx, axis=0).copy()
    # Acquirer price reflects deal premium (simplified)
    S_new[acquirer_idx, 0] += np.log(1 + 0.03)   # small synergy premium
    S_new[acquirer_idx, 1] += np.log(exchange_ratio + 1)  # share issuance
    return S_new

S_after_merger = apply_merger(S, 1, 2, exchange_ratio=0.5)
print(f'\nAfter M&A merger (asset 2 absorbed into asset 1):')
print(f'  Universe: {n_assets} → {S_after_merger.shape[0]} assets')
print(f'  State shape: {S.shape} → {S_after_merger.shape}')

## Why Standard Group Theory Fails

A **group** requires:
1. Closure: $T_1 \circ T_2 \in G$ ✓ (for fixed n)
2. Associativity: $(T_1 \circ T_2) \circ T_3 = T_1 \circ (T_2 \circ T_3)$ ✓
3. Identity element ✓
4. Inverse element ✓

But after a Mode III merger $T_\text{merger}: \mathbb{R}^{4d} \to \mathbb{R}^{3d}$, the split operator $T_\text{split}: \mathbb{R}^{3d} \to \mathbb{R}^{3d}$ **cannot be composed** with $T_\text{merger}$ because their domains don't match.

A **groupoid** relaxes closure — composition is only defined when domains match:
$$T_1 \circ T_2 \text{ defined iff } \text{target}(T_2) = \text{source}(T_1)$$

This is exactly the structure of our event sequence: each operator has a source dimension and target dimension, and composition requires they chain correctly.

In [ ]:
# Demonstrate groupoid composition: a sequence of events
from dataclasses import dataclass

@dataclass
class Event:
    name: str
    mode: str  # 'I', 'II', 'III'
    source_n: int
    target_n: int

def is_composable(e1: Event, e2: Event) -> bool:
    """e1 ∘ e2 is defined iff target(e2) == source(e1)"""
    return e2.target_n == e1.source_n

# Market event timeline (2022-2024 inspired)
events = [
    Event('MSFT+ATVI merger', 'III', source_n=500, target_n=499),
    Event('Fed +75bp hike',   'II',  source_n=499, target_n=499),
    Event('NVDA 10:1 split',  'I',   source_n=499, target_n=499),
    Event('ARM IPO',          'III', source_n=499, target_n=500),
    Event('SVB delisting',    'III', source_n=500, target_n=499),
    Event('Index rebalance',  'II',  source_n=499, target_n=499),
]

print('Event composition chain (groupoid validity check):')
print(f'{"Event":<25} {"Mode":<5} {"n_in":<6} {"n_out":<6} {"Composable with next?"}')
print('-' * 65)
for i, e in enumerate(events):
    if i < len(events) - 1:
        comp = is_composable(events[i+1], e)
        status = 'YES ✓' if comp else 'NO ✗ (domain mismatch!)'
    else:
        status = '(last event)'
    print(f'{e.name:<25} {e.mode:<5} {e.source_n:<6} {e.target_n:<6} {status}')

print('\nAll adjacent events compose correctly — groupoid structure is consistent.')

## Implementation Note

The full `EventOperator` class is in `events/operators.py`. Key methods:

- `stock_split_operator(ratio, d)` — Mode I
- `rate_hike_operator(shock_bps, n_assets, durations)` — Mode II  
- `merger_operator(acquirer, target, n_assets, d)` — Mode III
- `ipo_operator(ticker, d, n_assets)` — Mode III
- `compose(t1, t2)` — groupoid composition with domain checking

The Encoder E handles discrete events by maintaining an **event queue** $\mathcal{E}_t$ that is drained between encoder steps. When a Mode III event fires, the latent space $z \in \mathbb{R}^d$ is updated via a learned **event embedding** that absorbs the dimensional change without breaking the Markov structure.

This is a key architectural innovation: standard world models (DreamerV3, RSSM) assume a fixed state space. MicroWorld handles a *dynamically changing* investment universe.